# Medium Future (1-Hour) AI Training

This notebook trains the Transformer+GRU model to predict **1-hour price movements** (Medium Future). 

**Why?**
- **Avoids the Fee Trap:** 5-minute scalping requires ~87% accuracy to break even due to fees. 1-hour swings (1.5%+) easily cover fees.
- **Reduces Drawdown:** Filters out market noise and "fakeouts" that occur on lower timeframes.
- **Higher Win Rate:** Trends are more stable over 1-4 hours than 5-15 minutes.

In [1]:
# 1. Install Dependencies
import sys
!{sys.executable} -m pip install -q pybit pandas numpy torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 13.0 MB/s eta 0:00:0000:0100:01


In [5]:
# 2. Setup Environment
import os
import sys
from pathlib import Path

# Try to find the project root
possible_paths = [
    os.getcwd(),
    '/content/drive/MyDrive/trading_bot',
    '/Users/ariatajeri/trading_bot'
]

PROJECT_PATH = None

for path in possible_paths:
    if os.path.exists(os.path.join(path, 'training', 'innovative_trainer.py')):
        PROJECT_PATH = path
        break

if PROJECT_PATH:
    print(f"✅ Project found at: {PROJECT_PATH}")
    if PROJECT_PATH not in sys.path:
        sys.path.append(PROJECT_PATH)
    os.chdir(PROJECT_PATH)
else:
    print(f"❌ Project NOT found. Current Dir: {os.getcwd()}")
    print("If you are on Colab, please mount Drive:")
    print("from google.colab import drive")
    print("drive.mount('/content/drive')")
    
    # Attempt mount if in Colab
    try:
        from google.colab import drive
        print("Attempting to mount Drive...")
        drive.mount('/content/drive')
        # Re-check
        if os.path.exists('/content/drive/MyDrive/trading_bot'):
            PROJECT_PATH = '/content/drive/MyDrive/trading_bot'
            sys.path.append(PROJECT_PATH)
            os.chdir(PROJECT_PATH)
            print(f"✅ Project found after mount: {PROJECT_PATH}")
    except Exception as e:
        print(f"Mount failed: {e}")
        print("👉 Please switch to a LOCAL Kernel (top right) to run with local files.")

❌ Project NOT found. Current Dir: /content
If you are on Colab, please mount Drive:
from google.colab import drive
drive.mount('/content/drive')
Attempting to mount Drive...
Mount failed: mount failed
👉 Please switch to a LOCAL Kernel (top right) to run with local files.


In [ ]:
# 3. Configure for Medium Future (1-Hour)
from training.innovative_trainer import InnovativeTrainer, TrainerConfig
from pathlib import Path

config = TrainerConfig(
    # Data - optimized for 1H horizon
    sequence_length=72,       # 6 hours of context (72 * 5m)
    prediction_horizon=12,    # 1 hour ahead (12 * 5m)
    
    # Label thresholds for 1H swing (0.6% move covers fees easily)
    up_threshold=0.006,
    down_threshold=-0.006,
    use_adaptive_threshold=True,
    
    # Training Params - INCREASED FOR BETTER RESULTS
    pretrain_epochs=20,       # Increased from 5
    supervised_epochs=50,     # Increased from 10
    rl_episodes=50,           # Increased from 10
    
    # Paths
    checkpoint_dir=Path(PROJECT_PATH) / 'checkpoints',
    log_dir=Path(PROJECT_PATH) / 'logs'
)

print("Configuration Loaded: Medium Future (1-Hour Horizon) - PRODUCTION MODE")

ModuleNotFoundError: No module named 'training'

In [ ]:
# 4. Fetch & Process Data
import pandas as pd
import numpy as np
from pybit.unified_trading import HTTP
import time

def calculate_rsi(prices, period=14):
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def get_top_liquid_symbols(n=100):
    """Fetch top N USDT pairs by 24h turnover from Bybit"""
    print(f"Fetching top {n} liquid symbols from Bybit...")
    session = HTTP()
    try:
        resp = session.get_tickers(category="linear")
        if resp['retCode'] != 0:
            print("❌ Failed to fetch tickers")
            return ['BTCUSDT', 'ETHUSDT', 'SOLUSDT'] # Fallback
            
        items = resp['result']['list']
        scored = []
        for it in items:
            sym = it['symbol']
            if not sym.endswith('USDT'): continue
            turnover = float(it.get('turnover24h', 0))
            scored.append((turnover, sym))
            
        scored.sort(reverse=True)
        top_symbols = [s for _, s in scored[:n]]
        print(f"✅ Selected {len(top_symbols)} symbols: {top_symbols[:5]} ...")
        return top_symbols
    except Exception as e:
        print(f"❌ Error fetching symbols: {e}")
        return ['BTCUSDT', 'ETHUSDT', 'SOLUSDT'] # Fallback

def fetch_and_process_data(symbols=None, days=180): 
    if symbols is None:
        symbols = get_top_liquid_symbols(100)
        
    session = HTTP()
    all_data = []
    
    for i, symbol in enumerate(symbols):
        print(f"[{i+1}/{len(symbols)}] Fetching {days} days for {symbol}...")
        klines = []
        end_time = None
        target = 288 * days # 5m candles
        
        # Limit retries per symbol
        retries = 3
        while len(klines) < target and retries > 0:
            params = {'category': 'linear', 'symbol': symbol, 'interval': '5', 'limit': 1000}
            if end_time: params['end'] = end_time
            
            try:
                resp = session.get_kline(**params)
                if resp['retCode'] != 0: 
                    retries -= 1
                    time.sleep(1)
                    continue
                    
                batch = resp['result']['list']
                if not batch: break
                
                klines.extend(batch)
                end_time = int(batch[-1][0]) - 1
                time.sleep(0.1) # Rate limit protection
            except Exception as e:
                print(f"Error: {e}")
                retries -= 1
                time.sleep(1)
        
        if not klines: continue
        
        # Process DataFrame
        df = pd.DataFrame(klines, columns=['ts', 'open', 'high', 'low', 'close', 'vol', 'turn'])
        df = df.astype(float)
        df = df.sort_values('ts').reset_index(drop=True)
        
        # Feature Engineering (Must match main.py logic)
        df['returns'] = df['close'].pct_change()
        df['volatility'] = df['returns'].rolling(20).std()
        df['sma_10'] = df['close'].rolling(10).mean()
        df['sma_20'] = df['close'].rolling(20).mean()
        df['rsi'] = calculate_rsi(df['close'], 14)
        df['volume_ma'] = df['vol'].rolling(20).mean()
        
        # ATR
        df['tr'] = np.maximum(df['high'] - df['low'], 
                              np.maximum(abs(df['high'] - df['close'].shift(1)), 
                                         abs(df['low'] - df['close'].shift(1))))
        df['atr'] = df['tr'].rolling(14).mean()
        df['atr_pct'] = df['atr'] / df['close']
        
        # Momentum
        df['momentum_3'] = df['close'].pct_change(3)
        df['momentum_6'] = df['close'].pct_change(6)
        df['momentum_12'] = df['close'].pct_change(12)
        
        # Ratios
        df['volume_ratio'] = df['vol'] / (df['volume_ma'] + 1e-8)
        df['price_sma10_ratio'] = df['close'] / (df['sma_10'] + 1e-8) - 1
        df['price_sma20_ratio'] = df['close'] / (df['sma_20'] + 1e-8) - 1
        
        # MACD
        df['ema_12'] = df['close'].ewm(span=12).mean()
        df['ema_26'] = df['close'].ewm(span=26).mean()
        df['macd'] = df['ema_12'] - df['ema_26']
        df['macd_signal'] = df['macd'].ewm(span=9).mean()
        df['macd_hist'] = df['macd'] - df['macd_signal']
        
        # Bollinger Bands
        df['bb_mid'] = df['close'].rolling(20).mean()
        df['bb_std'] = df['close'].rolling(20).std()
        df['bb_upper'] = df['bb_mid'] + 2 * df['bb_std']
        df['bb_lower'] = df['bb_mid'] - 2 * df['bb_std']
        df['bb_position'] = (df['close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'] + 1e-8)
        df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / (df['bb_mid'] + 1e-8)
        
        all_data.append(df.dropna())
        
    if not all_data:
        return np.array([])
        
    # Combine
    combined = pd.concat(all_data, ignore_index=True)
    
    # Select Features
    feature_cols = [
        'close', 'returns', 'volatility', 'rsi', 'macd', 'macd_signal', 'macd_hist',
        'bb_position', 'bb_width', 'atr_pct', 'momentum_3', 'momentum_6', 'momentum_12',
        'volume_ratio', 'price_sma10_ratio', 'price_sma20_ratio'
    ]
    
    # Normalize (except close)
    for col in feature_cols[1:]:
        mean = combined[col].mean()
        std = combined[col].std()
        if std > 0:
            combined[col] = (combined[col] - mean) / std
            
    return combined[feature_cols].values.astype(np.float32)

# Fetch data for top 100 symbols
data = fetch_and_process_data(days=180)
print(f"Training Data Shape: {data.shape}")

In [ ]:
# 5. Run Training
import logging
import sys

# Force re-configuration of logging to ensure output appears in the notebook
root = logging.getLogger()
root.setLevel(logging.INFO)

# Clear existing handlers to avoid duplicates or silence
if root.handlers:
    for handler in root.handlers:
        root.removeHandler(handler)

# Add stdout handler
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
handler.setFormatter(formatter)
root.addHandler(handler)

# Explicitly set level for the trainer module
logging.getLogger("training.innovative_trainer").setLevel(logging.INFO)

print("✅ Logging configured. Starting Training Pipeline...")

trainer = InnovativeTrainer(config)

if len(data) == 0:
    print("❌ Error: No data fetched. Cannot train.")
else:
    results = trainer.train_full_pipeline(data)

    print("\nTraining Complete!")
    
    # Extract best loss correctly
    supervised_res = results.get('supervised', {})
    best_loss = supervised_res.get('best_val_loss')
    
    print(f"Best Validation Loss: {best_loss}")
    print(f"Model saved to: {config.checkpoint_dir}")
    
    # Print full results for debugging
    import json
    print("\nFull Results Summary:")
    print(json.dumps(results, indent=2, default=str))

In [ ]:
# 6. Download Model (Colab Only)
try:
    from google.colab import files
    import os
    
    # Define path to the best model
    model_path = config.checkpoint_dir / "transformer_best.pt"
    
    if os.path.exists(model_path):
        print(f"Found model at: {model_path}")
        print("Initiating download...")
        files.download(model_path)
    else:
        print(f"❌ Model not found at: {model_path}")
        print("Check if training completed successfully.")
except ImportError:
    print("ℹ️ Not running in Google Colab.")
    print(f"Model is saved locally at: {config.checkpoint_dir / 'transformer_best.pt'}")